# Практическая работа № 5. Классификатор и анализ метрик
**Цель:** построить модель бинарной классификации и провести анализ качества с учётом цены ошибки.  
**Вариант B:** подобрать порог классификации под заданную цену ошибки и обосновать выбор метрики.

## 0. Настройки
Замените `CSV_PATH`, `TARGET`, `COST_FP`, `COST_FN` на значения из задания. Если `CSV_PATH = None`, используются синтетические данные с дисбалансом ~20/80.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score,
                             accuracy_score, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score)

CSV_PATH = None        # например "data.csv"
TARGET = "target"      # имя целевого столбца (0/1)
COST_FP = 1.0          # цена ложноположительного
COST_FN = 10.0         # цена ложноотрицательного
SEED = 42

## 1. Данные и разбиение
Порог подбирается на **валидационной** выборке, итоговая оценка — на **тестовой**, чтобы не было утечки.

In [ ]:
def load_data():
    if CSV_PATH:
        df = pd.read_csv(CSV_PATH).dropna()
        return df.drop(columns=[TARGET]), df[TARGET]
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=5000, n_features=15, n_informative=6,
                               weights=[0.8, 0.2], flip_y=0.03,
                               class_sep=0.9, random_state=SEED)
    return pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])]), pd.Series(y)

X, y = load_data()
print("Размер:", X.shape)
print("Доля положительного класса:", round(y.mean(), 3))

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=SEED)
print(len(X_train), len(X_val), len(X_test))

## 2. Модель и вероятности классов (`predict_proba`)

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

p_val = model.predict_proba(X_val)[:, 1]
p_test = model.predict_proba(X_test)[:, 1]
p_test[:5].round(3)

## 3. ROC- и PR-кривые, AUC

In [ ]:
roc_auc = roc_auc_score(y_test, p_test)
pr_auc = average_precision_score(y_test, p_test)
fpr, tpr, _ = roc_curve(y_test, p_test)
prec, rec, _ = precision_recall_curve(y_test, p_test)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
ax[0].plot([0, 1], [0, 1], "--", color="gray")
ax[0].set(xlabel="FPR (доля ложных тревог)", ylabel="TPR (recall)", title="ROC-кривая")
ax[0].legend()
ax[1].plot(rec, prec, label=f"PR-AUC (AP) = {pr_auc:.3f}")
ax[1].axhline(y_test.mean(), ls="--", color="gray", label="baseline = доля класса 1")
ax[1].set(xlabel="Recall", ylabel="Precision", title="PR-кривая")
ax[1].legend()
plt.tight_layout(); plt.show()

**Комментарий.** AUC не зависит от порога и оценивает качество ранжирования объектов по вероятности. При дисбалансе классов PR-кривая информативнее ROC: она учитывает precision и сравнивается с базовым уровнем, равным доле положительного класса.

## 4. Подбор порога под цену ошибки
Суммарная цена ошибок: `COST_FP · FP + COST_FN · FN`. Теоретический ориентир для порога: `COST_FP / (COST_FP + COST_FN)`.  
Перебираем пороги на валидации и берём тот, при котором цена минимальна.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)

def total_cost(y_true, p, t):
    tn, fp, fn, tp = confusion_matrix(y_true, (p >= t).astype(int)).ravel()
    return COST_FP * fp + COST_FN * fn

costs = [total_cost(y_val, p_val, t) for t in thresholds]
best_t = thresholds[int(np.argmin(costs))]
print(f"Теоретический порог: {COST_FP/(COST_FP+COST_FN):.3f}")
print(f"Выбранный порог (валидация): {best_t:.2f}")

plt.figure(figsize=(6, 4))
plt.plot(thresholds, costs)
plt.axvline(0.5, ls="--", color="gray", label="0.5")
plt.axvline(best_t, color="red", label=f"выбранный = {best_t:.2f}")
plt.xlabel("Порог"); plt.ylabel("Суммарная цена ошибок (валидация)")
plt.title("Выбор порога"); plt.legend(); plt.show()

## 5. Сравнение метрик на тесте: порог 0,5 и подобранный порог

In [ ]:
rows, cms = [], {}
for name, t in [("Порог 0.5", 0.5), (f"Порог {best_t:.2f}", best_t)]:
    pred = (p_test >= t).astype(int)
    cm = confusion_matrix(y_test, pred)
    cms[name] = cm
    tn, fp, fn, tp = cm.ravel()
    rows.append({"Порог": name,
                 "Accuracy": accuracy_score(y_test, pred),
                 "Precision": precision_score(y_test, pred),
                 "Recall": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred),
                 "FP": fp, "FN": fn,
                 "Цена ошибок": COST_FP * fp + COST_FN * fn})
res = pd.DataFrame(rows).round(3)
res

### Матрицы ошибок

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
for a, (name, cm) in zip(ax, cms.items()):
    ConfusionMatrixDisplay(cm, display_labels=["0 (отриц.)", "1 (полож.)"]).plot(
        ax=a, cmap="Blues", colorbar=False)
    a.set(title=f"Матрица ошибок, {name}",
          xlabel="Предсказанный класс", ylabel="Истинный класс")
plt.tight_layout(); plt.show()

## 6. Вывод
*(Числа ниже — для синтетических данных; после подстановки своих данных и цен перепишите под свои результаты.)*

1. **Выбор метрики.** Если ложноотрицательный результат дороже ложноположительного, главной метрикой должен быть **recall** при приемлемой precision. Accuracy при дисбалансе вводит в заблуждение: при пороге 0,5 она высокая, но модель пропускает большую часть положительных объектов (recall ≈ 0,29). F1 — вспомогательная метрика, она считает FP и FN равноценными, поэтому для выбора порога использована суммарная цена ошибок.
2. **Обоснование порога.** Порог снижен с 0,5 до значения, минимизирующего `COST_FP·FP + COST_FN·FN` на валидации (около 0,09, что совпадает с теоретическим `COST_FP/(COST_FP+COST_FN)`).
3. **Результат.** Recall вырос примерно с 0,29 до 0,94, суммарная цена ошибок на тесте упала в ~2,5 раза. Платой стал рост числа ложных тревог (FP), снижение precision и accuracy. При заданной цене ошибки этот компромисс оправдан.
4. **Если цены FP и FN равны**, допустимо оставить порог 0,5: оснований смещать порог нет, а дисбаланс умеренный.